# Deep dive into Text Generation Inference with LLMs

到目前为止，我们已经研究了与一系列离散任务（如文本分类或摘要）相关的转换器体系结构。然而，大型语言模型主要用于文本生成，这也是我们将在本章探讨的内容。

在本页中，我们将探索LLM推理背后的核心概念，全面了解这些模型如何生成文本以及推理过程中涉及的关键组件。

## Understanding the Basics

让我们从基础开始。推理是使用经过训练的语言模型（LLM）根据给定的输入提示生成类似人类的文本的过程。语言模型利用其从训练中获得的知识，逐词地构建响应。该模型利用从数十亿参数中学到的概率来预测并生成序列中的下一个标记。这种顺序生成使得语言模型能够生成连贯且与上下文相关的文本。

## The Role of Attention

注意力机制使大型语言模型能够理解上下文并生成连贯的回应。在预测下一个单词时，并非句子中的每个单词都具有同等的重要性——例如，在“法国的首都……”这个句子中，“法国”和“首都”这两个词对于确定接下来应该出现“巴黎”这一单词至关重要。这种专注于相关信息的能力就是我们所说的注意力。

This ability to focus on relevant information is what we call attention.

这种确定最相关词汇以预测下一个词的这一过程已被证明非常有效。尽管训练大型语言模型（即预测下一个词）的基本原理自 BERT 和 GPT-2 以来一直大致保持不变，但在扩展神经网络规模以及让注意力机制适用于更长且更复杂的序列方面，以及在降低成本方面，已经取得了显著的进步。

简而言之，注意力机制是让大型语言模型能够生成既连贯又具有上下文意识的文本的关键所在。它使现代大型语言模型与以往的语言模型有了显著的区别。

### Context Length and Attention Span

上下文长度是模型在生成响应时一次可以考虑的令牌的最大数量。

### The Art of Prompting

当我们将信息传递给LLM时，我们以一种引导LLM生成所需输出的方式构建我们的输入。这叫做提示(prompting).

了解语言模型如何处理信息有助于我们设计出更有效的提示。由于该模型的主要任务是通过分析每个输入词的重要性来预测下一个词，所以您输入序列的措辞就变得至关重要了。

仔细设计prompt可以更容易地引导LLM生成所需的输出。

## The Two-Phase Inference Process

既然我们已经了解了基本构成要素，接下来让我们深入探讨一下大型语言模型是如何生成文本的。这个过程可以分为两个主要阶段：预填充和解码。这两个阶段就像一条流水线一样协同运作，每个阶段都对生成连贯的文本起着至关重要的作用。

### The Prefill Phase

预填充阶段就像烹饪的准备阶段——在这个阶段，所有的原料都被加工好并准备好。这个阶段包括三个关键步骤：

- 分词(Tokenization): 将input分解为token
- embedding conversion：将token转换为数学表示（向量）
- Initial Processing：通过模型的神经网络运行这些嵌入，以创建对上下文的丰富理解

这一阶段的计算量很大，因为它需要一次性处理所有的token。可以将其理解为在开始回复之前先完整地阅读并理解整个段落。

### The Decode Phase

在预填充阶段处理完输入信息后，我们便进入解码阶段——这是实际文本生成的环节。该模型会逐个生成一个token，我们称之为自回归过程（其中每个新词元都取决于所有之前的token）。

解码阶段涉及每个新令牌的几个关键步骤：

1. 注意力计算：回顾所有先前的标记以理解上下文
2. 概率计算：确定每个可能的下一个词出现的可能性
3. token选择：根据这些概率来挑选下一个token
4. 继续与否的检查：决定是否继续生成或停止生成

这一阶段对内存的消耗较大，因为模型需要记录所有先前生成的标记及其之间的关系。

## Sampling Strategies(抽样策略)

既然我们已经了解了该模型生成文本的原理，接下来让我们探讨一下如何控制这一生成过程的各种方式。就像作家可能会在追求创意与追求精确度之间做出选择一样，我们也可以调整模型进行token选择的方式

在这个空间中，您可以亲自操作使用 SmolLM2 进行基本的解码过程（请记住，该模型的解码会一直进行，直至遇到 EOS 标记）：